# Lecture: Understanding KL divergence

**Objective**: Fully understand the Kullback-Leibler (KL) divergence as a
*directed* measure of how one probability distribution differs from another,
and see why its **direction (asymmetry)** matters in policy-gradient methods
such as TRPO and PPO.

In policy-gradient methods the KL divergence measures the change in behaviour
when we update from an old policy $\pi_{old}$ to a new policy $\pi_{new}$. A
small divergence means minor changes; a large divergence implies the agent's
behaviour may have shifted drastically.

Two different KL terms show up in the theory, and they use the arguments in
**opposite order** &mdash; this notebook makes that explicit:

- the **relative policy performance bound** (TRPO / Kakade-Langford) penalises
  $D_{KL}(\pi_{new}\,\|\,\pi_{old})$,
- the quantity PPO actually monitors as `approx_kl` is
  $D_{KL}(\pi_{old}\,\|\,\pi_{new})$.

Because $D_{KL}(P\,\|\,Q) \neq D_{KL}(Q\,\|\,P)$, these are *not* the same
number. Keeping them apart is the whole point of Exercise&nbsp;3.

### Exercise 1: Implementing KL divergence with minor stability tricks

Below you see the code for turning raw, unscaled policy outputs (logits) into a
probability distribution (the `softmax()` method)

$$ \text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}} $$

and the code for computing the `kl_divergence()`:

$$ D_{KL}(P\,\|\,Q) = \sum_x P(x)\,\log \frac{P(x)}{Q(x)} $$

There are two numerical subtleties in the implementation. Find them and make
sure you understand *why* each is needed:

1. **`softmax` subtracts `np.max(logits)`** before exponentiating. This does not
   change the result mathematically (it cancels in numerator and denominator)
   but prevents `np.exp` from overflowing for large logits.
2. **`kl_divergence` clips only `q` (the denominator)** to a small `epsilon` so
   we never divide by &mdash; or take the log of &mdash; zero. We deliberately do
   **not** clip `p`: instead we use the convention $0\,\log 0 = 0$ via
   `np.where`, so terms where $P(x)=0$ contribute nothing. Clipping `p` would
   change its mass and the result would no longer be a proper KL divergence.

Read the documentation of `np.clip()` and `np.where()` to confirm what they do.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def softmax(logits):
    """Turn raw logits into a probability distribution (numerically stable)."""
    logits = np.array(logits, dtype=float)
    e_x = np.exp(logits - np.max(logits))   # subtract max -> no overflow
    return e_x / e_x.sum()


def kl_divergence(p, q, epsilon=1e-12, base=np.e):
    """KL divergence D_KL(P || Q) = sum_x P(x) * log(P(x) / Q(x)).

    Parameters
    ----------
    p, q : array-like
        Discrete probability distributions of the same shape. ``p`` is the
        reference/weighting distribution, ``q`` the one it is compared against.
    epsilon : float
        Lower clip for ``q`` only, so we never divide by / take log of zero.
    base : float
        Log base. ``np.e`` gives nats, ``2`` gives bits.
    """
    p = np.asarray(p, dtype=float)
    q = np.clip(np.asarray(q, dtype=float), epsilon, None)   # clip q only
    # 0 * log(0) is defined as 0 -> only sum over the support of p
    terms = np.where(p > 0, p * np.log(p / q), 0.0)
    return terms.sum() / np.log(base)


### Exercise 2: Try it out!

You now have three policies: a reference policy $\pi_{old}$ and two candidate
new policies $A$ and $B$. We want to measure how far each new policy is from the
reference.

#### Task 1
Try the given numbers as well as other numbers in each policy. Get a feeling for
how changing probabilities impacts the KL divergence. Use the bar-chart cell to
visualise the policies.

#### Task 2
The output is given in *nats*. What is a nat, and is it a reasonable unit here?
Compare it to *bits* by passing `base=2` to `kl_divergence` &mdash; when would
you prefer one over the other?

In [ ]:
# Reference policy (before the update)
ref_policy = np.array([0.6, 0.3, 0.1])   # pi_old

# New Policy A: similar to the reference
policy_a = np.array([0.55, 0.35, 0.10])  # small shift

# New Policy B: significantly different
policy_b = np.array([0.2, 0.4, 0.4])     # large shift


In [ ]:
# D_KL(pi_old || pi_new) -- the direction PPO monitors as approx_kl
kl_a = kl_divergence(ref_policy, policy_a)
kl_b = kl_divergence(ref_policy, policy_b)

print(f"KL(old || A) = {kl_a:.4f} nats  = {kl_divergence(ref_policy, policy_a, base=2):.4f} bits")
print(f"KL(old || B) = {kl_b:.4f} nats  = {kl_divergence(ref_policy, policy_b, base=2):.4f} bits")


### Exercise 3: The KL divergence is asymmetric (the key property)

This is the most important thing to understand about the KL divergence:

$$ D_{KL}(P\,\|\,Q) \neq D_{KL}(Q\,\|\,P) $$

It is **not** a distance. Swapping the arguments generally gives a different
number, because the sum is weighted by the *first* argument $P$: it emphasises
the regions where $P$ puts its mass.

The two directions have names and different behaviour:

- **Forward KL** $D_{KL}(P\,\|\,Q)$ is *mass-covering*: it heavily penalises a
  $Q$ that is near zero where $P$ is large (the $\log(P/Q)$ term blows up).
- **Reverse KL** $D_{KL}(Q\,\|\,P)$ is *mode-seeking*.

The cell below computes both directions for policies $A$ and $B$.

In [ ]:
for name, new in [("A", policy_a), ("B", policy_b)]:
    fwd = kl_divergence(ref_policy, new)   # D_KL(old || new)  -> approx_kl in PPO
    rev = kl_divergence(new, ref_policy)   # D_KL(new || old)  -> performance bound
    print(f"Policy {name}:  KL(old || new) = {fwd:.4f}   "
          f"KL(new || old) = {rev:.4f}   "
          f"ratio = {rev / fwd:.2f}")


#### Which direction appears where in TRPO / PPO?

| Term | Direction | Role |
|------|-----------|------|
| Relative policy performance bound (TRPO theory) | $D_{KL}(\pi_{new}\,\|\,\pi_{old})$ | Penalty that *guarantees* monotonic improvement |
| TRPO trust-region constraint / PPO `approx_kl` | $D_{KL}(\pi_{old}\,\|\,\pi_{new})$ | Practically estimated early-stopping / diagnostic signal |

The performance-bound penalty weights by the **new** policy, whereas the
practical constraint (and PPO's `approx_kl`) weights by the **old** policy,
whose samples we already have &mdash; that is why the practical estimator is
cheap to compute from a rollout. The asymmetry above is exactly why these two
are different numbers.

In `81-CartPole-PPO.ipynb` you will see `approx_kl` printed during training:
that is the $D_{KL}(\pi_{old}\,\|\,\pi_{new})$ column.

### Exercise 4: Non-negativity and the danger of $q \to 0$

Two more properties worth verifying by hand:

1. **Non-negativity (Gibbs' inequality):** $D_{KL}(P\,\|\,Q) \ge 0$, with
   equality **iff** $P = Q$. So $D_{KL}(\pi_{old}\,\|\,\pi_{old}) = 0$.
2. **$q \to 0$ blows up:** if the new policy assigns almost zero probability to
   an action the old policy still takes, the $\log(P/Q)$ term explodes. This is
   precisely the instability PPO guards against by **clipping** the probability
   ratio &mdash; it prevents the update from pushing any $q$ towards zero too
   aggressively.

In [ ]:
# 1) KL is zero for identical distributions, positive otherwise
print(f"KL(old || old) = {kl_divergence(ref_policy, ref_policy):.4f}  (== 0)")

# 2) Let the new policy starve action 3 (which old still takes with p=0.1)
for q3 in [0.05, 0.01, 1e-3, 1e-6]:
    new = np.array([0.6, 1.0 - 0.6 - q3, q3])
    print(f"q(action 3) = {q3:<8}  KL(old || new) = {kl_divergence(ref_policy, new):.4f} nats")

In [ ]:
labels = ['Action 1', 'Action 2', 'Action 3']
x = np.arange(len(labels))
width = 0.25

plt.bar(x - width, ref_policy, width, label='Reference (old)')
plt.bar(x,         policy_a,   width, label='Policy A (similar)')
plt.bar(x + width, policy_b,   width, label='Policy B (different)')

plt.ylabel('Action Probabilities')
plt.title('Comparison of Stochastic Policies')
plt.xticks(x, labels)
plt.ylim(0, 1)
plt.legend()
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.show()


### Outlook: KL divergence between continuous distributions

So far everything was discrete (a categorical policy over 3 actions). For
**continuous control** &mdash; where PPO parametrises the policy as a Gaussian
$\mathcal{N}(\mu, \sigma^2)$ &mdash; the sum becomes an integral, but for two
Gaussians the KL has a closed form:

$$ D_{KL}\big(\mathcal{N}(\mu_1,\sigma_1^2)\,\|\,\mathcal{N}(\mu_2,\sigma_2^2)\big)
   = \log\frac{\sigma_2}{\sigma_1}
   + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2}
   - \frac{1}{2} $$

The cell below implements this and shows how the divergence grows as the new
policy's mean drifts away from the old one &mdash; the continuous analogue of
the discrete experiment above.

In [ ]:
def kl_gaussian(mu1, sigma1, mu2, sigma2):
    """Closed-form KL divergence D_KL(N1 || N2) for two 1D Gaussians (in nats)."""
    return (np.log(sigma2 / sigma1)
            + (sigma1**2 + (mu1 - mu2)**2) / (2 * sigma2**2)
            - 0.5)


# Old policy: N(0, 1). Drift the new policy's mean away and watch KL grow.
mu_old, sigma_old = 0.0, 1.0
mus = np.linspace(-3, 3, 200)
kls = [kl_gaussian(mu_old, sigma_old, m, sigma_old) for m in mus]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Left: KL as a function of the new mean
ax1.plot(mus, kls)
ax1.axvline(mu_old, color='grey', ls='--', alpha=0.7, label='old mean')
ax1.set_xlabel(r'new mean $\mu_2$')
ax1.set_ylabel(r'$D_{KL}(\mathcal{N}_{old} \| \mathcal{N}_{new})$ [nats]')
ax1.set_title('KL grows quadratically with mean shift')
ax1.legend()
ax1.grid(True, ls='--', alpha=0.5)

# Right: the two densities for one example shift
xs = np.linspace(-6, 6, 400)
gauss = lambda x, mu, s: np.exp(-0.5 * ((x - mu) / s)**2) / (s * np.sqrt(2 * np.pi))
mu_new = 1.5
ax2.plot(xs, gauss(xs, mu_old, sigma_old), label=r'old $\mathcal{N}(0,1)$')
ax2.plot(xs, gauss(xs, mu_new, sigma_old), label=fr'new $\mathcal{{N}}({mu_new},1)$')
ax2.set_title(f'KL(old || new) = {kl_gaussian(mu_old, sigma_old, mu_new, sigma_old):.3f} nats')
ax2.set_xlabel('action')
ax2.set_ylabel('density')
ax2.legend()
ax2.grid(True, ls='--', alpha=0.5)

plt.tight_layout()
plt.show()
